In [1]:
import pandas as pd
import os
from pathlib import Path

import sys
sys.path.append('../src') # add src directory to path

from utils import format_discussion
from setfit import SetFitModel, SetFitTrainer
from sentence_transformers.losses import CosineSimilarityLoss

In [13]:
dataset = pd.read_csv('../data/opinion_dataset.csv')

In [3]:
dataset.shape

(60, 2)

In [14]:
#create test and train set (8 samples per class for training and the rest is testing)
train_data = dataset.groupby('target').apply(lambda x: x.sample(8, random_state=42)).reset_index(drop=True)
#test set is the rest of the data
test_data = dataset.drop(train_data.index).reset_index(drop=True)
test_data.shape, train_data.shape

/var/folders/06/25_1mnt97hq60zy5672y3qzw0000gn/T/ipykernel_35435/868968334.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data = dataset.groupby('target').apply(lambda x: x.sample(8, random_state=42)).reset_index(drop=True)


((44, 2), (16, 2))

In [15]:
#converting to hugging face format
from datasets import Dataset

#rename target column to label (since its required by setfit)
train_data = train_data.rename(columns={'target': 'label'})
test_data = test_data.rename(columns={'target': 'label'})

train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

this is the model used: https://huggingface.co/blog/setfit

In [16]:
# Load SetFit model from Hub
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-mpnet-base-v2")

# Create trainer
trainer = SetFitTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss_class=CosineSimilarityLoss,
    batch_size=16,
    num_iterations=40, # Number of text pairs to generate for contrastive learning
    num_epochs=1 # Number of epochs to use for contrastive learning
)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/var/folders/06/25_1mnt97hq60zy5672y3qzw0000gn/T/ipykernel_35435/3916725541.py:5: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

In [17]:
trainer.train()
metrics = trainer.evaluate()

***** Running training *****


  Num unique pairs = 1280
  Batch size = 16
  Num epochs = 1
/Users/andreacristiano/Desktop/VNIVERSITA/MAGISTRALE 2 ANNO/Big Data analytics/stance-detection-eu-parliaments/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,0.298500
50,0.041700


***** Running evaluation *****


In [10]:
metrics

{'accuracy': 0.9545454545454546}

In [11]:
model.save_pretrained( save_directory='./../models/setfit-fewshot-opinions-1')

In [12]:
model.push_to_hub("andreacristiano/stancedetection")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/andreacristiano/stancedetection/commit/abaa0948f75c29a3f52be3de155138e96a3e8f33', commit_message='Push model using huggingface_hub.', commit_description='', oid='abaa0948f75c29a3f52be3de155138e96a3e8f33', pr_url=None, repo_url=RepoUrl('https://huggingface.co/andreacristiano/stancedetection', endpoint='https://huggingface.co', repo_type='model', repo_id='andreacristiano/stancedetection'), pr_revision=None, pr_num=None)

In [12]:
model1 = SetFitModel.from_pretrained("andreacristiano/stancedetection")

config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config_setfit.json:   0%|          | 0.00/91.0 [00:00<?, ?B/s]

model_head.pkl:   0%|          | 0.00/7.12k [00:00<?, ?B/s]

In [13]:
model1.metrics

AttributeError: 'SetFitModel' object has no attribute 'metrics'